# ACRS on AppWorld: O1–O4 + G1 with open models on a free or Colab Pro GPU

Jenisha T · M S Ramaiah University of Applied Sciences · 24ETRP720001

`Runtime → Run all`. The notebook:
1. picks the largest open models that fit this GPU;
2. serves them with vLLM;
3. runs the official-style baseline, full ACRS and leave-one-out ablations on AppWorld **dev**;
4. scores everything with AppWorld's official evaluator (TGC/SGC) and runs paired tests.

Every config gets the same `FEW_SHOT` worked examples in its prompt: solved tasks from AppWorld's **train** split (the AppWorld authors' own demonstration tasks first), read from the downloaded data at run time.

**Rules (AppWorld):** tune on train/dev only; test splits run once per final configuration. Do not publish unencrypted AppWorld task or API content. Experiment outputs stay in this runtime; only aggregate summaries leave it.

In [ ]:
import os, subprocess, json, time, sys
T_START = time.time()
# Dev tasks per config (first PILOT_N); 0 = the whole split (57 tasks).
PILOT_N = int(os.environ.get('ACRS_PILOT_N', '0'))
BRANCH = os.environ.get('ACRS_BRANCH', 'main')   # repository branch to run
# Record each step's decisions (O1-O4, G1) in the episode logs; they stay in this runtime's output.
TRACE_FLAG = '--trace' if os.environ.get('ACRS_TRACE', '1') == '1' else ''
RUN_SMOKE = os.environ.get('ACRS_SMOKE', '0') == '1'  # smoke validated on Kaggle 2026-11-05
BUDGET_H = float(os.environ.get('ACRS_BUDGET_H', '10.5'))  # stop starting new configs after this; a new session resumes
# Worked examples from the train split in every config's prompt (same for all configs); 0 = none.
FEW_SHOT = int(os.environ.get('ACRS_FEW_SHOT', '3'))
FS = f'_fs{FEW_SHOT}' if FEW_SHOT else ''   # part of the experiment name, so runs never mix
# Hard stop between tasks (Kaggle ends sessions at 12 h); finished episodes are kept and scored.
os.environ['ACRS_DEADLINE'] = str(T_START + 3600 * float(os.environ.get('ACRS_HARD_STOP_H', '11.0')))
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
WORK = '/kaggle/working' if IN_KAGGLE else '/content'
os.chdir(WORK)
mem = int(subprocess.run(['nvidia-smi','--query-gpu=memory.total','--format=csv,noheader,nounits'],capture_output=True,text=True).stdout.split()[0])
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True).stdout.strip()
if mem >= 39000:   LARGE, SMALL, UL, US, LEN = 'Qwen/Qwen3-32B-AWQ', 'Qwen/Qwen3-4B-AWQ', 0.62, 0.22, 24576
elif mem >= 22000: LARGE, SMALL, UL, US, LEN = 'Qwen/Qwen3-14B-AWQ', 'Qwen/Qwen3-4B-AWQ', 0.58, 0.28, 16384
# T4 (16 GB, compute 7.5): 8B-AWQ weights take ~5.7 GB, so the large tier needs ~2/3 of memory for a usable
# KV cache; eager mode frees the CUDA-graph reserve, and the small tier is the 0.6B model.
else: LARGE, SMALL, UL, US, LEN = 'Qwen/Qwen3-8B-AWQ', 'Qwen/Qwen3-0.6B', 0.66, 0.26, 10240
EAGER = mem < 22000
# Colab's T4 VM has ~12.7 GB RAM; each AppWorld worker holds its own app databases, so keep few workers there.
WORKERS = 2 if EAGER else 8
# Kaggle 2xT4 (30 GB RAM): one server per GPU, a stronger small tier, more workers.
NGPU = len(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout.strip().splitlines())
if NGPU >= 2 and EAGER:
    LARGE, SMALL, UL, US, LEN, WORKERS = 'Qwen/Qwen3-8B-AWQ', 'Qwen/Qwen3-4B-AWQ', 0.90, 0.90, 16384, 4
# Configs without O2 never call the small tier: the large model then runs on both GPUs, with more workers.
WORKERS_2X = 6 if NGPU >= 2 else WORKERS
print(NGPU, 'x', gpu, mem, 'MiB ->', LARGE, '+', SMALL)

In [ ]:
os.chdir(WORK)
!rm -rf acrs-appworld && git clone -q --depth 1 -b {BRANCH} https://github.com/joyjeni/acrs-appworld.git
%cd acrs-appworld
!git log --oneline -1
# Resume across sessions: attach this notebook's previous output (Add Input > Notebook Output); finished
# episodes are copied back and --resume skips them (their recorded per-task results are scored).
import glob, shutil
for f in glob.glob('/kaggle/input/**/experiments/results/appworld/*.episodes.jsonl', recursive=True):
    dst = os.path.join('experiments/results/appworld', os.path.basename(f))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        shutil.copy(f, dst); print('resumed', os.path.basename(f), sum(1 for _ in open(dst)), 'episodes')
# vLLM (pydantic 2) and AppWorld 0.1.3 (pydantic 1) cannot share an environment:
# each gets its own venv; they talk over HTTP only.
!pip -q install uv 2>&1 | tail -1
!uv venv -q --clear --python 3.12 /opt/vllm && uv pip install -q --python /opt/vllm/bin/python vllm 2>&1 | tail -2
!/opt/vllm/bin/python -c "import vllm; print('vllm', vllm.__version__)"
!uv venv -q --clear --python 3.11 /opt/aw && uv pip install -q --python /opt/aw/bin/python "appworld==0.1.3.post1" numpy scipy pytest 2>&1 | tail -2
!/opt/aw/bin/python -c "import appworld; print('appworld', appworld.__version__)"
# AppWorld data gets its own root: `download data` would otherwise overwrite ./data (the ToolBench bundle).
os.environ['APPWORLD_ROOT'] = AW_ROOT = f'{WORK}/appworld_root'; os.makedirs(AW_ROOT, exist_ok=True)
!/opt/aw/bin/appworld install 2>&1 | tail -1
!cd $AW_ROOT && /opt/aw/bin/appworld download data 2>&1 | tail -1
# Fail fast (before any GPU time) if the train split does not give FEW_SHOT usable worked examples.
if FEW_SHOT:
    chk = subprocess.run(['/opt/aw/bin/python', 'experiments/check_demos.py', '--k', str(FEW_SHOT)], capture_output=True, text=True)
    print(chk.stdout[-2000:], chk.stderr[-500:])
    assert chk.returncode == 0, 'not enough usable train examples: set ACRS_FEW_SHOT=0 or check the AppWorld data'
!/opt/aw/bin/python -m pytest -q tests 2>&1 | tail -2


In [ ]:
def serve(model, port, util, maxlen, gpu_id=None):
    log = open(f'/tmp/vllm_{port}.log', 'w')
    env = dict(os.environ, **({'CUDA_VISIBLE_DEVICES': str(gpu_id)} if gpu_id is not None else {}))
    return subprocess.Popen(['/opt/vllm/bin/python','-m','vllm.entrypoints.openai.api_server','--model',model,'--port',str(port),
        '--gpu-memory-utilization',str(util),'--max-model-len',str(maxlen),'--disable-log-requests', '--dtype', 'half' if EAGER else 'auto'] + (['--enforce-eager'] if EAGER else []), stdout=log, stderr=log, env=env)

def wait(port, proc, timeout=1800):
    import urllib.request
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None: break  # server died: fail fast with its log
        try:
            urllib.request.urlopen(f'http://localhost:{port}/v1/models', timeout=5); return True
        except Exception: time.sleep(10)
    raise RuntimeError(open(f'/tmp/vllm_{port}.log').read()[-3000:])

def stop(proc):
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try: proc.wait(60)
        except Exception: proc.kill()
        time.sleep(5)

# 'mixed': large model on GPU 0 (8001), small on GPU 1 (8002), for configs that route between tiers (O2).
# 'large2': the large model on both GPUs (8001, 8003), for configs without O2, so the second GPU is not idle.
SERVERS = {}
def ensure_servers(mode):
    mode = mode if NGPU >= 2 else 'mixed'
    if SERVERS.get('mode') == mode: return
    if 'large' not in SERVERS:
        SERVERS['large'] = serve(LARGE, 8001, UL, LEN, 0 if NGPU >= 2 else None); wait(8001, SERVERS['large'])
    stop(SERVERS.pop('second', None))
    if mode == 'large2':
        SERVERS['second'] = serve(LARGE, 8003, US, LEN, 1); wait(8003, SERVERS['second'])
    else:
        SERVERS['second'] = serve(SMALL, 8002, US, LEN, 1 if NGPU >= 2 else None); wait(8002, SERVERS['second'])
    SERVERS['mode'] = mode
    print('servers:', mode, flush=True)

def specs(mode):
    large = f'{LARGE}@http://localhost:8001/v1' + ('|http://localhost:8003/v1' if mode == 'large2' and NGPU >= 2 else '')
    return large, f'{SMALL}@http://localhost:8002/v1'

NO_ROUTER = {'baseline', 'o4_only', 'o4_fidelity_only'}
def mode_for(cfg):
    return 'large2' if cfg in NO_ROUTER and NGPU >= 2 else 'mixed'

# The large model starts now; what runs on the second GPU is chosen per config (ensure_servers).
SERVERS['large'] = serve(LARGE, 8001, UL, LEN, 0 if NGPU >= 2 else None); wait(8001, SERVERS['large'])
L, S = specs('mixed')
print('large model up')

## 1 · Smoke test: 5 dev tasks, baseline vs full (about 10 min)

In [ ]:
if RUN_SMOKE:
    ensure_servers('mixed')
    for cfg in ['baseline', 'full']:
        !/opt/aw/bin/python experiments/appworld_run.py --split dev --config {cfg} --limit 5 --workers {WORKERS} --tag smoke --small "{S}" --large "{L}" --context-limit {LEN} --few-shot {FEW_SHOT} --resume 2>&1 | tail -25

## 2 · Dev pilot: baseline, O4 alone, full ACRS and leave-one-out ablations

In [ ]:
# Most informative first: the O4 pairs (baseline vs o4_only is cheap; full vs no_o4 is the integrated test),
# then O3 (the core contribution), then the rest.
CONFIGS = ['baseline', 'o4_only', 'full', 'no_o4', 'no_o3', 'no_g1', 'no_o1', 'no_o2']
import shutil
for cfg in CONFIGS:
    if (time.time() - T_START) / 3600 > BUDGET_H:
        print('time budget reached; skipping', cfg); continue
    mode = mode_for(cfg); ensure_servers(mode); L_CFG, S_CFG = specs(mode)
    N_WORKERS = WORKERS_2X if mode == 'large2' else WORKERS
    t0 = time.time()
    !/opt/aw/bin/python experiments/appworld_run.py --split dev --config {cfg} --limit {PILOT_N} --workers {N_WORKERS} --small "{S_CFG}" --large "{L_CFG}" --context-limit {LEN} --few-shot {FEW_SHOT} {TRACE_FLAG} --resume > /tmp/run_{cfg}.log 2>&1
    print(cfg, f'{(time.time()-t0)/60:.1f} min'); print(open(f'/tmp/run_{cfg}.log').read()[-600:])
    # Archive aggregate summaries after every config, so a quota cut-off keeps finished results.
    shutil.make_archive(f'{WORK}/acrs_appworld_summaries', 'zip', 'experiments/results/appworld')


## 3 · Results and paired significance (McNemar exact test on per-task success, bootstrap CI)

In [ ]:
import glob, random, math
import pandas as pd
from scipy.stats import binomtest
R = 'experiments/results/appworld'
def load(cfg):
    s = json.load(open(f'{R}/acrs_{cfg}{FS}_s0_dev.summary.json'))
    ev = json.load(open(f'{R}/acrs_{cfg}{FS}_s0_dev.evaluation.json'))
    per = {t: v.get('success', False) for t, v in ev.get('individual', {}).items()}
    return s, per
rows, per = [], {}
for cfg in CONFIGS:
    try: s, p = load(cfg)
    except FileNotFoundError: continue
    per[cfg] = p; a = s['official']
    rows.append({'config': cfg, 'TGC': a.get('task_goal_completion'), 'SGC': a.get('scenario_goal_completion'),
                 'tokens(M)': round(s['tokens']/1e6, 2), 'small/large': f"{s['router']['small']}/{s['router']['large']}",
                 'certified/abstain': f"{s['consensus']['certified']}/{s['consensus']['abstain_disagree']+s['consensus']['abstain_all_failed']}",
                 'prompt tok/task': round(s.get('promptTokens', 0) / max(1, s.get('tasksAttempted', 1))),
                 'hist tok/step': round(s.get('meanHistoryTokens', 0)), 'refetch': s.get('refetchCalls'), 'logins': s.get('loginCalls'),
                 'login exempt': s['consensus'].get('exempt_steps', 0), 'policy denials': s['policyDenials'], 'prov denials': s.get('provDenials'), 'fidelity': round(s['meanFidelity'], 3), 'failed eps': s['episodesFailed']})
display(pd.DataFrame(rows))
def mcnemar(a, b):
    ks = sorted(set(per[a]) & set(per[b])); b01 = sum(per[a][k] and not per[b][k] for k in ks); b10 = sum(per[b][k] and not per[a][k] for k in ks)
    return b01, b10, (binomtest(b01, b01+b10, 0.5).pvalue if b01+b10 else 1.0)
def boot(a, b, n=5000):
    ks = sorted(set(per[a]) & set(per[b])); d = [int(per[a][k]) - int(per[b][k]) for k in ks]; rng = random.Random(0)
    m = sorted(sum(rng.choice(d) for _ in d)/len(d) for _ in range(n)); return round(m[int(.025*n)], 4), round(m[int(.975*n)], 4)
for other in [c for c in per if c != 'full']:
    w, l, p = mcnemar('full', other)
    print(f'full vs {other:9s}: full-only wins {w}, other-only wins {l}, McNemar p={p:.4f}, TGC diff 95% CI {boot("full", other)}')
# O4 (H4.1, H4.2): prompt tokens per task, paired over tasks (Wilcoxon), and task success (McNemar).
from scipy.stats import wilcoxon
def task_prompt_tokens(cfg):
    rows = [json.loads(l) for l in open(f'{R}/acrs_{cfg}{FS}_s0_dev.episodes.jsonl')]
    return {e['task_id']: e['usage']['prompt_tokens'] for e in rows if 'usage' in e}
PAIRS = []
for a, b in [('o4_only', 'baseline'), ('full', 'no_o4')]:
    if a not in per or b not in per: continue
    ta, tb = task_prompt_tokens(a), task_prompt_tokens(b); ks = sorted(set(ta) & set(tb))
    if not ks: continue
    pw = wilcoxon([ta[k] for k in ks], [tb[k] for k in ks]).pvalue if any(ta[k] != tb[k] for k in ks) else 1.0
    med = lambda t: sorted(t[k] for k in ks)[len(ks) // 2]
    w, l, p = mcnemar(a, b)
    print(f'O4: {a} vs {b} on {len(ks)} tasks: prompt tokens/task median {med(ta):,} vs {med(tb):,} '
          f'(Wilcoxon p={pw:.4g}); success {a}-only {w}, {b}-only {l} (McNemar p={p:.3f})')
    PAIRS.append({'a': a, 'b': b, 'tasks': len(ks), 'median_prompt_tokens': [med(ta), med(tb)], 'wilcoxon_p': pw,
                  'a_only_wins': w, 'b_only_wins': l, 'mcnemar_p': p})
# Aggregates only (no AppWorld task content), for the dashboard and the thesis tables.
json.dump({'configs': rows, 'o4_pairs': PAIRS, 'few_shot': FEW_SHOT, 'tasks_per_config': PILOT_N or 57,
           'models': {'large': LARGE, 'small': SMALL}, 'branch': BRANCH}, open(f'{WORK}/viva_results.json', 'w'),
          indent=1, default=str)
print('wrote', f'{WORK}/viva_results.json')


In [ ]:
import shutil
shutil.make_archive(f'{WORK}/acrs_appworld_summaries', 'zip', R)
print('download:', f'{WORK}/acrs_appworld_summaries.zip')